In [6]:
import os
import random
import shutil
import numpy as np
from pathlib import Path
from tqdm import tqdm

In [7]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from torchvision.datasets import ImageFolder
from PIL import Image

In [8]:
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix
from ultralytics import YOLO

In [9]:
CONFIG = {
    "dataset_path":     "TrashBox_train_set",      # Root folder with class subfolders
    "output_path":      "outputs/yolov8",       # Where to save results
    "labeled_ratio":    0.1,                    # 10% labeled, 90% unlabeled
    "num_epochs":       50,                     # Training epochs
    "batch_size":       16,                     # Batch size
    "image_size":       224,                    # Input image size
    "learning_rate":    0.001,                  # Learning rate
    "confidence_threshold": 0.95,              # FixMatch confidence threshold
    "num_classes":      6,                      # Number of trash categories
    "device":           "cuda" if torch.cuda.is_available() else "cpu",
    "seed":             42,
}

In [10]:
print(os.listdir(CONFIG["dataset_path"]))

['cardboard', 'e-waste', 'glass', 'medical', 'metal', 'paper', 'plastic']


In [11]:
print("Dataset path:", os.path.abspath(CONFIG["dataset_path"]))
print("Output path:", os.path.abspath(CONFIG["output_path"]))

Dataset path: c:\Users\LIONESS\Desktop\e-waste\TrashBox\TrashBox_train_set
Output path: c:\Users\LIONESS\Desktop\e-waste\TrashBox\outputs\yolov8


In [12]:
print(os.listdir(CONFIG["dataset_path"]))

['cardboard', 'e-waste', 'glass', 'medical', 'metal', 'paper', 'plastic']


In [13]:
# Trash class names
CLASS_NAMES = ["cardboard", "e-waste", "glass", "medical", "metal", "paper", "plastic"]


In [14]:
print(f"Using device: {CONFIG['device']}")
os.makedirs(CONFIG["output_path"], exist_ok=True)
random.seed(CONFIG["seed"])
torch.manual_seed(CONFIG["seed"])


Using device: cpu


In [15]:
def split_labeled_unlabeled(dataset_path, labeled_ratio=0.1, output_path="outputs"):
    labeled = os.path.join(output_path, "labeled")
    unlabeled = os.path.join(output_path, "unlabeled")
    val = os.path.join(output_path, "val")

    for split_directory in [labeled, unlabeled, val]:
        os.makedirs(split_directory, exist_ok=True)

    print("Dataset path:", os.path.abspath(dataset_path))
    print("Output path:", os.path.abspath(output_path))
    print("Classes found:", os.listdir(dataset_path))

    for class_name in os.listdir(dataset_path):
        class_path = os.path.join(dataset_path, class_name)

        if not os.path.isdir(class_path):
            continue

        images = [
            f for f in os.listdir(class_path)
            if f.lower().endswith((".jpg", ".jpeg", ".png", ".webp"))
        ]

        print(f"\nClass: {class_name}")
        print(f"Images found: {len(images)}")

        if len(images) == 0:
            print(f"Skipping {class_name}, no images found.")
            continue

        random.shuffle(images)

        n_labeled = max(1, int(len(images) * labeled_ratio))
        n_val = max(1, int(len(images) * 0.1))

        if n_labeled + n_val > len(images):
            n_val = max(0, len(images) - n_labeled)

        n_unlabeled = len(images) - n_labeled - n_val

        labeled_images = images[:n_labeled]
        val_images = images[n_labeled:n_labeled + n_val]
        unlabeled_images = images[n_labeled + n_val:]

        for split, split_images in [
            ("labeled", labeled_images),
            ("val", val_images),
            ("unlabeled", unlabeled_images),
        ]:
            split_class_dir = os.path.join(output_path, split, class_name)
            os.makedirs(split_class_dir, exist_ok=True)

            for img in split_images:
                src = os.path.join(class_path, img)
                dst = os.path.join(split_class_dir, img)
                shutil.copy2(src, dst)

        print(f"{class_name}: {n_labeled} labeled | {n_val} val | {n_unlabeled} unlabeled")

    print(f"\nSplit complete. Saved to: {output_path}")
    return labeled, unlabeled, val

In [16]:
import os

print(os.listdir("TrashBox_train_set"))
for class_name in os.listdir("TrashBox_train_set"):
    class_path = os.path.join("TrashBox_train_set", class_name)
    if os.path.isdir(class_path):
        images = [f for f in os.listdir(class_path) if f.lower().endswith((".jpg", ".jpeg", ".png", ".webp"))]
        print(class_name, len(images))

['cardboard', 'e-waste', 'glass', 'medical', 'metal', 'paper', 'plastic']
cardboard 1930
e-waste 2406
glass 2022
medical 1565
metal 2068
paper 2156
plastic 2135


In [17]:
if os.path.exists("outputs/yolov8"):
    shutil.rmtree("outputs/yolov8")

CONFIG = {
    "dataset_path": "TrashBox_train_set",
    "output_path": "outputs/yolov8",
    "labeled_ratio": 0.1,
}

split_labeled_unlabeled(
    dataset_path=CONFIG["dataset_path"],
    labeled_ratio=CONFIG["labeled_ratio"],
    output_path=CONFIG["output_path"]
)

Dataset path: c:\Users\LIONESS\Desktop\e-waste\TrashBox\TrashBox_train_set
Output path: c:\Users\LIONESS\Desktop\e-waste\TrashBox\outputs\yolov8
Classes found: ['cardboard', 'e-waste', 'glass', 'medical', 'metal', 'paper', 'plastic']

Class: cardboard
Images found: 1930
cardboard: 193 labeled | 193 val | 1544 unlabeled

Class: e-waste
Images found: 2406
e-waste: 240 labeled | 240 val | 1926 unlabeled

Class: glass
Images found: 2022
glass: 202 labeled | 202 val | 1618 unlabeled

Class: medical
Images found: 1565
medical: 156 labeled | 156 val | 1253 unlabeled

Class: metal
Images found: 2068
metal: 206 labeled | 206 val | 1656 unlabeled

Class: paper
Images found: 2156
paper: 215 labeled | 215 val | 1726 unlabeled

Class: plastic
Images found: 2135
plastic: 213 labeled | 213 val | 1709 unlabeled

Split complete. Saved to: outputs/yolov8


('outputs/yolov8\\labeled', 'outputs/yolov8\\unlabeled', 'outputs/yolov8\\val')

In [19]:
print(CONFIG)

{'dataset_path': 'TrashBox_train_set', 'output_path': 'outputs/yolov8', 'labeled_ratio': 0.1}


Fixmatch Augmentation

In [20]:
CONFIG = {
    "dataset_path": "TrashBox_train_set",
    "output_path": "outputs/yolov8",
    "labeled_ratio": 0.1,
    "num_epochs": 50,
    "batch_size": 16,
    "image_size": 224,
    "learning_rate": 0.001,
    "confidence_threshold": 0.95,
    "num_classes": 7,
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "seed": 42,
}

In [21]:
#Weak augmentation of the labeled data
from torchvision import transforms

weak_transform = transforms.Compose([
    transforms.Resize((CONFIG["image_size"], CONFIG["image_size"])),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])


In [23]:
#Strong augmentation of the unlabeled data
strong_transform = transforms.Compose([
    transforms.Resize((CONFIG["image_size"], CONFIG["image_size"])),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4, hue=0.1),
    transforms.RandomGrayscale(p=0.2),
    transforms.RandomAffine(degrees=30, translate=(0.1, 0.1)),
    transforms.RandomErasing(p=0.5, scale=(0.02, 0.2)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

In [24]:
val_transform = transforms.Compose([
    transforms.Resize((CONFIG["image_size"], CONFIG["image_size"])),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

In [25]:
class UnlabeledDataset(Dataset):
    def __init__(self, root_dir):
        self.image_paths = []
        for class_dir in Path(root_dir).iterdir():
            if class_dir.is_dir():
                for img_path in class_dir.glob("*"):
                    if img_path.suffix.lower() in ['.jpg', '.jpeg', '.png', '.webp']:
                        self.image_paths.append(str(img_path))
                        
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, index):
        image = Image.open(self.image_paths[index]).convert("RGB")
        weak = weak_transform(image)
        strong = strong_transform(image)
        return weak, strong

Fixmatch trainer

In [49]:
"""
    FixMatch Semi-Supervised Training:
    - Trains on labeled data with cross-entropy loss
    - Generates pseudo-labels on unlabeled data using weak augmentation
    - Only keeps pseudo-labels above confidence threshold
    - Trains on those pseudo-labels using strong augmentation
    """

class FixMatchTrainer:
    def __init__(self, model, labeled_loader, unlabeled_loader, val_loader, config):
        self.model = model.to(config["device"])
        self.labeled_loader = labeled_loader
        self.unlabeled_loader = unlabeled_loader
        self.val_loader = val_loader
        self.config = config
        self.device = config["device"]
        
        self.optimizer = optim.Adam(model.parameters(), lr=config["learning_rate"])
        self.scheduler = optim.lr_scheduler.CosineAnnealingLR(self.optimizer, T_max=config["num_epochs"])
        self.criterion = nn.CrossEntropyLoss()
        
        self.train_losses = []
        self.val_accuracies = []
        
    def train_epoch(self):
        self.model.train()
        total_loss = 0
        labeled_iter = iter(self.labeled_loader)
        
        for (weak_u, strong_u) in tqdm(self.unlabeled_loader, desc="Training", leave=False):
            #Labeled batch
            try:
                images_l, labels_l = next(labeled_iter)
            except StopIteration:
                labeled_iter = iter(self.labeled_loader)
                images_l, labels_l = next(labeled_iter)
            
            images_l = images_l.to(self.device)
            labels_l = labels_l.to(self.device)
            
            #Supervised loss on the labeled data
            logits_l = self.model(images_l)
            loss_labeled = self.criterion(logits_l, labels_l)
            
            #The unlabeled batch (Fixmatch)
            weak_u = weak_u.to(self.device)
            strong_u = strong_u.to(self.device)
            
            with torch.no_grad():
                probs_weak = torch.softmax(self.model(weak_u), dim=1)
                confidence, pseudo_labels = probs_weak.max(dim=1)
                
            #only keeping high confidence pseudo labels
            mask = confidence >= self.config["confidence_threshold"]
            
            loss_unlabeled = torch.tensor(0.0, device=self.device)
            if mask.sum() > 0:
                logits_strong = self.model(strong_u[mask])
                loss_unlabeled = self.criterion(logits_strong, pseudo_labels[mask])
                
            #The combined loss
            loss = loss_labeled + loss_unlabeled
            self.optimizer.zero_grad()
            loss.backward()
            self.optimizer.step()
            
            total_loss += loss.item()
            
        self.scheduler.step()
        return total_loss / len(self.unlabeled_loader)
    
    def validate(self):
        self.model.eval()
        correct, total = 0, 0
        all_preds, all_labels = [], []
        
        with torch.no_grad():
            for images, labels in self.val_loader:
                images = images.to(self.device)
                labels = labels.to(self.device)
                outputs = self.model(images)
                _, predicted = outputs.max(1)
                correct += predicted.eq(labels).sum().item()
                total   += labels.size(0)
                all_preds.extend(predicted.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())

        accuracy = 100.0 * correct / total
        return accuracy, all_preds, all_labels

    def train(self):
        print("\n" + "="*50)
        print("  Starting FixMatch Semi-Supervised Training")
        print("="*50)
        best_acc = 0

        for epoch in range(self.config["num_epochs"]):
            loss = self.train_epoch()
            acc, preds, labels = self.validate()

            self.train_losses.append(loss)
            self.val_accuracies.append(acc)

            print(f"Epoch [{epoch+1:3d}/{self.config['num_epochs']}] "
                  f"Loss: {loss:.4f} | Val Acc: {acc:.2f}%")

            if acc > best_acc:
                best_acc = acc
                torch.save(self.model.state_dict(),
                           f"{self.config['output_path']}/best_fixmatch_model.pth")
                print(f"  ✅ New best model saved ({acc:.2f}%)")

        print(f"\nBest Validation Accuracy: {best_acc:.2f}%")
        self.plot_training()

        # Final report
        acc, preds, labels = self.validate()
        print("\nClassification Report:")
        print(classification_report(labels, preds,
                                    target_names=CLASS_NAMES))

    def plot_training(self):
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
        ax1.plot(self.train_losses)
        ax1.set_title("Training Loss")
        ax1.set_xlabel("Epoch")
        ax2.plot(self.val_accuracies, color="green")
        ax2.set_title("Validation Accuracy (%)")
        ax2.set_xlabel("Epoch")
        plt.tight_layout()
        plt.savefig(f"{self.config['output_path']}/training_curves.png")
        print(f"Training curves saved.")
        
        
    

In [50]:
# STEP 5: YOLOV8 DETECTION TRAINING
def train_yolov8_detection(dataset_yaml_path):
    """
    Trains YOLOv8 for object detection on trash images.

    dataset_yaml_path: path to a YOLO-format dataset.yaml file.

    Example dataset.yaml:
        path: /absolute/path/to/dataset
        train: images/train
        val: images/val
        nc: 6
        names: ['plastic', 'glass', 'metal', 'cardboard', 'organic', 'other']
    """
    print("\n" + "="*50)
    print("  Starting YOLOv8 Detection Training")
    print("="*50)

    # Load pretrained YOLOv8 nano (fastest) or use yolov8s.pt / yolov8m.pt for accuracy
    model = YOLO("yolov8n.pt")  # Pretrained on COCO — transfer learning

    results = model.train(
        data=dataset_yaml_path,
        epochs=CONFIG["num_epochs"],
        imgsz=CONFIG["image_size"],
        batch=CONFIG["batch_size"],
        name="trash_yolov8",
        project=CONFIG["output_path"],
        device=CONFIG["device"],
        pretrained=True,           # Transfer learning from COCO weights
        patience=10,               # Early stopping
        augment=True,              # Built-in augmentations
        verbose=True,
    )

    print("\nYOLOv8 Training Complete!")
    print(f"Results saved to: {CONFIG['output_path']}/trash_yolov8")
    return model


In [51]:
def run_yolov8_inference(model_path, image_path):
    """Run YOLOv8 inference on a single image or folder."""
    model = YOLO(model_path)
    results = model.predict(
        source=image_path,
        conf=0.25,       # Confidence threshold
        save=True,       # Save annotated images
        show_labels=True,
        show_conf=True,
    )

    for result in results:
        boxes = result.boxes
        for box in boxes:
            cls   = int(box.cls[0])
            conf  = float(box.conf[0])
            label = CLASS_NAMES[cls] if cls < len(CLASS_NAMES) else f"class_{cls}"
            print(f"  Detected: {label} (confidence: {conf:.2f})")

    return results

In [52]:
# STEP 6: OPTIONAL PREDICTION MODULE
class WastePredictionHead(nn.Module):
    """
    Optional regression head to predict:
    - Bin fill level (0.0 to 1.0)
    - Waste volume estimate
    Add this on top of your trained FixMatch backbone.
    """

    def __init__(self, feature_dim=512):
        super().__init__()
        self.regressor = nn.Sequential(
            nn.Linear(feature_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
            nn.Sigmoid()     # Output: 0.0 (empty) to 1.0 (full)
        )

    def forward(self, features):
        return self.regressor(features)

In [53]:
# MAIN — RUN EVERYTHING
if __name__ == "__main__":

    print("\n🗑️  Trash Detection & Classification — YOLOv8 + FixMatch")
    print("="*55)

    # PART A: FixMatch Semi-Supervised Classification
    print("\n📂 Step 1: Splitting dataset into labeled/unlabeled...")
    labeled_dir, unlabeled_dir, val_dir = split_labeled_unlabeled(
        dataset_path=CONFIG["dataset_path"],
        labeled_ratio=CONFIG["labeled_ratio"],
        output_path=CONFIG["output_path"]
    )

    print("\n📦 Step 2: Loading datasets...")
    labeled_dataset   = ImageFolder(labeled_dir,   transform=weak_transform)
    unlabeled_dataset = UnlabeledDataset(unlabeled_dir)
    val_dataset       = ImageFolder(val_dir,       transform=val_transform)

    labeled_loader   = DataLoader(labeled_dataset,   batch_size=CONFIG["batch_size"],
                                   shuffle=True,  num_workers=2)
    unlabeled_loader = DataLoader(unlabeled_dataset, batch_size=CONFIG["batch_size"],
                                   shuffle=True,  num_workers=2)
    val_loader       = DataLoader(val_dataset,       batch_size=CONFIG["batch_size"],
                                   shuffle=False, num_workers=2)

    print(f"  Labeled:   {len(labeled_dataset)} images")
    print(f"  Unlabeled: {len(unlabeled_dataset)} images")
    print(f"  Val:       {len(val_dataset)} images")

    print("\n🧠 Step 3: Building model (ResNet50 + Transfer Learning)...")
    model = models.resnet50(pretrained=True)
    model.fc = nn.Linear(model.fc.in_features, CONFIG["num_classes"])

    print("\n🚀 Step 4: Training FixMatch...")
    trainer = FixMatchTrainer(model, labeled_loader, unlabeled_loader, val_loader, CONFIG)
    trainer.train()

    # -----------------------------------------------------------
    # PART B: YOLOv8 Detection (requires YOLO dataset.yaml)
    # -----------------------------------------------------------
    # Uncomment below once you have a YOLO-format dataset with bounding boxes
    #
    # print("\n🔍 Step 5: Training YOLOv8 for detection...")
    # yolo_model = train_yolov8_detection("dataset/dataset.yaml")
    #
    # print("\n🖼️  Step 6: Running inference...")
    # run_yolov8_inference(
    #     model_path=f"{CONFIG['output_path']}/trash_yolov8/weights/best.pt",
    #     image_path="test_images/"
    # )

    print("\n✅ Done! Check outputs/ folder for saved models and results.")



🗑️  Trash Detection & Classification — YOLOv8 + FixMatch

📂 Step 1: Splitting dataset into labeled/unlabeled...
Dataset path: c:\Users\LIONESS\Desktop\e-waste\TrashBox\TrashBox_train_set
Output path: c:\Users\LIONESS\Desktop\e-waste\TrashBox\outputs\yolov8
Classes found: ['cardboard', 'e-waste', 'glass', 'medical', 'metal', 'paper', 'plastic']

Class: cardboard
Images found: 1930
cardboard: 193 labeled | 193 val | 1544 unlabeled

Class: e-waste
Images found: 2406
e-waste: 240 labeled | 240 val | 1926 unlabeled

Class: glass
Images found: 2022
glass: 202 labeled | 202 val | 1618 unlabeled

Class: medical
Images found: 1565
medical: 156 labeled | 156 val | 1253 unlabeled

Class: metal
Images found: 2068
metal: 206 labeled | 206 val | 1656 unlabeled

Class: paper
Images found: 2156
paper: 215 labeled | 215 val | 1726 unlabeled

Class: plastic
Images found: 2135
plastic: 213 labeled | 213 val | 1709 unlabeled

Split complete. Saved to: outputs/yolov8

📦 Step 2: Loading datasets...
  Label

AttributeError: partially initialized module 'torch._dynamo' has no attribute 'decorators' (most likely due to a circular import)

In [55]:
import os, sys, torch

print("torch version:", torch.__version__)
print("torch file:", torch.__file__)
print("python version:", sys.version)
print("cwd:", os.getcwd())
print("files in cwd:", os.listdir())

torch version: 2.7.1+cpu
torch file: c:\Users\LIONESS\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\__init__.py
python version: 3.12.3 (tags/v3.12.3:f6650f9, Apr  9 2024, 14:05:25) [MSC v.1938 64 bit (AMD64)]
cwd: c:\Users\LIONESS\Desktop\e-waste\TrashBox
files in cwd: ['.git', 'CITATION.cff', 'Dinov2.ipynb', 'eda.ipynb', 'outputs', 'README.md', 'sam_vit_b_01ec64.pth', 'TrashBox_description.pdf', 'TrashBox_train_dataset_subfolders', 'TrashBox_train_set', 'Yolov8 fixmatch trash.ipynb']
